# Efficiency in Practice II: Introduction to Profiling with Python

## 1) Time Tracking

**Keeping track of execution time**: Monitor runtime by seconds of your Python code.

Recommended methods for algorithm benchmarking:

- Use `time.time()` for prototyping
- Use `timeit` for small microbenchmarks when needed
- Use `time.perf_counter()` for high-resolution wall-clock timing
- **Repeat measurements** and report `median` (and optionally `min`) instead of a single run
- Keep setup work outside the timed block
- Compare across input sizes (`n`) to observe growth trends

In [1]:
import time


def time_func_p(func, *args):
    start_time = time.time()
    func(*args)
    end_time = time.time()
    elapsed_time = (end_time - start_time) * 1000  # in milliseconds
    return elapsed_time

In [2]:
import timeit as pytime


def time_func(func, *args, repeat=10, number=10000):
    durations = pytime.repeat(lambda: func(*args), repeat=repeat, number=number)
    best_time = min(durations) / number * 1000  # Convert to milliseconds
    print(f"Best time over {repeat} runs of {number} executions: {best_time:.5f} ms")

In [2]:
import statistics
import time


def benchmark(func, *args, repeats=7, warmup=1, **kwargs):
    for _ in range(warmup):
        func(*args, **kwargs)

    times_ms = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        func(*args, **kwargs)
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000)

    return {
        "median_ms": statistics.median(times_ms),
        "min_ms": min(times_ms),
        "max_ms": max(times_ms),
    }

## Examples

In [1]:
def linear_sum(n):
    return sum(range(n))

In [3]:
def quadratic_pair_count(n):
    c = 0
    for i in range(n):
        for j in range(n):
            c += (i + j) & 1
    return c

### Time Benchmarking

In [4]:
sizes = [300, 600, 1200]
print("n    linear median (ms)    quadratic median (ms)")
for n in sizes:
    t_lin = benchmark(linear_sum, n)
    t_quad = benchmark(quadratic_pair_count, n)
    print(f"{n:<5}{t_lin['median_ms']:>12.3f}{t_quad['median_ms']:>24.3f}")

n    linear median (ms)    quadratic median (ms)
300         0.002                   3.947
600         0.006                  17.972
1200        0.014                  74.623


## 2) A Deterministic Profiler: `cProfile`

You can find more about Python profilers [here](https://docs.python.org/3/library/profile.html).

In [12]:
import cProfile

cProfile.run("quadratic_pair_count(1200)")

         349 function calls (346 primitive calls) in 0.078 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.005    0.005    0.005    0.005 2587997395.py:1(quadratic_pair_count)
        1    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
      2/1    0.005    0.003    0.024    0.024 <string>:1(<module>)
        1    0.000    0.000    0.073    0.073 asyncio.py:206(_handle_events)
        1    0.000    0.000    0.000    0.000 attrsettr.py:43(__getattr__)
        1    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        1    0.000    0.000    0.000    0.000 base_events.py:1932(_add_callback)
        1    0.000    0.000    0.000    0.000 base_events.py:768(time)
        1    0.000    0.000    0.000    0.000 contextlib.py:108(__init__)
        1    0.000    0.000    0.000    0.000 contextlib.py:136(__enter__)
        1    0.000    0.000    0.000    0.000 cont